In [2]:
print("hello")

hello


Import libraries

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

Import data

In [4]:
df= pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")
df["date"]=pd.to_datetime(df["date"])
df=(df.sort_values("date").reset_index(drop=True))
print("shape:", df.shape)
display(df.head())

shape: (1887, 4)


,date,pm1,pm2_5,pm10
0,2016-08-25,144.026083,187.599837,269.346300
1,2016-08-27,121.881565,159.537520,230.975440
2,2016-08-28,98.196718,124.696523,171.362190
3,2016-08-29,54.083770,63.915300,78.042140
4,2016-09-07,127.947061,168.501950,245.585955


Create 7 days lag

In [5]:
for lag in range(1, 8):
    df[f"pm2_5_lag_{lag}"] = df["pm2_5"].shift(lag)
    df[f"pm10_lag_{lag}"] = df["pm10"].shift(lag)
    df[f"pm1_lag_{lag}"]= df["pm1"].shift(lag)

Create forecasting target

In [6]:
df["target_pm10"] = df["pm10"].shift(-1)

Remove unusable rows

In [7]:
df_model= df.dropna().copy()
print("original rows:", len(df))
print("usable rows:", len(df_model))

display(df_model.head())

original rows: 1887
usable rows: 1879


,date,pm1,pm2_5,pm10,pm2_5_lag_1,pm10_lag_1,pm1_lag_1,pm2_5_lag_2,pm10_lag_2,pm1_lag_2,...,pm2_5_lag_5,pm10_lag_5,pm1_lag_5,pm2_5_lag_6,pm10_lag_6,pm1_lag_6,pm2_5_lag_7,pm10_lag_7,pm1_lag_7,target_pm10
7,2016-09-12,105.117225,132.785217,182.608250,29.042917,32.999425,26.360702,63.253007,83.520250,51.507960,...,124.696523,171.362190,98.196718,159.537520,230.975440,121.881565,187.599837,269.346300,144.026083,166.891925
8,2016-09-18,92.885558,119.215280,166.891925,132.785217,182.608250,105.117225,29.042917,32.999425,26.360702,...,63.915300,78.042140,54.083770,124.696523,171.362190,98.196718,159.537520,230.975440,121.881565,130.395945
9,2016-09-19,76.851917,96.302953,130.395945,119.215280,166.891925,92.885558,132.785217,182.608250,105.117225,...,168.501950,245.585955,127.947061,63.915300,78.042140,54.083770,124.696523,171.362190,98.196718,147.140180
10,2016-09-20,80.535347,104.368733,147.140180,96.302953,130.395945,76.851917,119.215280,166.891925,92.885558,...,63.253007,83.520250,51.507960,168.501950,245.585955,127.947061,63.915300,78.042140,54.083770,145.790225
11,2016-09-21,83.568385,106.495640,145.790225,104.368733,147.140180,80.535347,96.302953,130.395945,76.851917,...,29.042917,32.999425,26.360702,63.253007,83.520250,51.507960,168.501950,245.585955,127.947061,137.984515


## Rolling Stats

Import data and sorting

In [8]:
rolling_df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

rolling_df["date"] = pd.to_datetime(
    rolling_df["date"]
)

rolling_df = (
    rolling_df.sort_values("date").reset_index(drop=True))

display(rolling_df.head())

,date,pm1,pm2_5,pm10
0,2016-08-25,144.026083,187.599837,269.346300
1,2016-08-27,121.881565,159.537520,230.975440
2,2016-08-28,98.196718,124.696523,171.362190
3,2016-08-29,54.083770,63.915300,78.042140
4,2016-09-07,127.947061,168.501950,245.585955


Create target

In [9]:
rolling_df["target_pm10"]= (rolling_df["pm10"]).shift(-1)

Create rolling features with 3 days and 7 days mean and std

In [10]:
for pollutant in ["pm2_5", "pm10","pm1"]:

    rolling_df[f"{pollutant}_rolling_mean_3"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=3)
        .mean()
    )

    rolling_df[f"{pollutant}_rolling_mean_7"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=7)
        .mean()
    )

    rolling_df[f"{pollutant}_rolling_std_3"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=3)
        .std()
    )

    rolling_df[f"{pollutant}_rolling_std_7"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=7)
        .std()
    )

adding minn and max too 

In [12]:
for pollutant in ["pm2_5", "pm10","pm1"]:

    rolling_df[f"{pollutant}_rolling_min_3"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=3)
        .min()
    )

    rolling_df[f"{pollutant}_rolling_min_7"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=7)
        .min()
    )

    rolling_df[f"{pollutant}_rolling_max_3"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=3)
        .max()
    )

    rolling_df[f"{pollutant}_rolling_max_7"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=7)
        .max()
    )

In [14]:
print("Shape:", rolling_df.shape)

display(
    rolling_df[
        [
            "date",
            "pm2_5",
            "pm10",
            "pm1",
            "pm2_5_rolling_mean_3",
            "pm2_5_rolling_mean_7",
            "pm10_rolling_mean_3",
            "pm10_rolling_mean_7",
            "pm1_rolling_mean_3",
            "pm1_rolling_mean_7",
            "target_pm10",
        ]
    ].head(10)
)

Shape: (1887, 29)


,date,pm2_5,pm10,pm1,pm2_5_rolling_mean_3,pm2_5_rolling_mean_7,pm10_rolling_mean_3,pm10_rolling_mean_7,pm1_rolling_mean_3,pm1_rolling_mean_7,target_pm10
0,2016-08-25,187.599837,269.346300,144.026083,NaN,NaN,NaN,NaN,NaN,NaN,230.975440
1,2016-08-27,159.537520,230.975440,121.881565,NaN,NaN,NaN,NaN,NaN,NaN,171.362190
2,2016-08-28,124.696523,171.362190,98.196718,NaN,NaN,NaN,NaN,NaN,NaN,78.042140
3,2016-08-29,63.915300,78.042140,54.083770,157.277960,NaN,223.894643,NaN,121.368122,NaN,245.585955
4,2016-09-07,168.501950,245.585955,127.947061,116.049781,NaN,160.126590,NaN,91.387351,NaN,83.520250
5,2016-09-10,63.253007,83.520250,51.507960,119.037924,NaN,164.996762,NaN,93.409183,NaN,32.999425
6,2016-09-11,29.042917,32.999425,26.360702,98.556752,NaN,135.716115,NaN,77.846264,NaN,182.608250
7,2016-09-12,132.785217,182.608250,105.117225,86.932624,113.792436,120.701877,158.833100,68.605241,89.143408,166.891925
8,2016-09-18,119.215280,166.891925,92.885558,75.027047,105.961776,99.709308,146.441950,60.995296,83.585000,130.395945
9,2016-09-19,96.302953,130.395945,76.851917,93.681138,100.201456,127.499867,137.287162,74.787828,79.442713,147.140180


Remove unusable rows 

In [15]:
rolling_model_df = (
    rolling_df
    .dropna()
    .reset_index(drop=True)
)

print("Original rows:", len(rolling_df))
print("Usable rows:", len(rolling_model_df))

Original rows: 1887
Usable rows: 1879


EWMA

Import data and sorting

In [16]:
ewma_df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

ewma_df["date"] = pd.to_datetime(
    ewma_df["date"]
)

ewma_df = (
    ewma_df.sort_values("date").reset_index(drop=True)
)

Create target

In [17]:
ewma_df["target_pm10"] = (ewma_df["pm10"].shift(-1))

Create features

In [18]:
for pollutant in ["pm2_5", "pm10","pm1"]:

    ewma_df[f"{pollutant}_ewma_3"] = (
        ewma_df[pollutant]
        .shift(1)
        .ewm(span=3, adjust=False)
        .mean()
    )

    ewma_df[f"{pollutant}_ewma_7"] = (
        ewma_df[pollutant]
        .shift(1)
        .ewm(span=7, adjust=False)
        .mean()
    )

In [20]:
display(
    ewma_df[
        [
            "date",
            "pm2_5",
            "pm10",
            "pm1",
            "pm2_5_ewma_3",
            "pm2_5_ewma_7",
            "pm10_ewma_3",
            "pm10_ewma_7",
            "pm1_ewma_3",
            "pm1_ewma_7",
            "target_pm10",
        ]
    ].head(10)
)

,date,pm2_5,pm10,pm1,pm2_5_ewma_3,pm2_5_ewma_7,pm10_ewma_3,pm10_ewma_7,pm1_ewma_3,pm1_ewma_7,target_pm10
0,2016-08-25,187.599837,269.346300,144.026083,NaN,NaN,NaN,NaN,NaN,NaN,230.975440
1,2016-08-27,159.537520,230.975440,121.881565,187.599837,187.599837,269.346300,269.346300,144.026083,144.026083,171.362190
2,2016-08-28,124.696523,171.362190,98.196718,173.568678,180.584258,250.160870,259.753585,132.953824,138.489953,78.042140
3,2016-08-29,63.915300,78.042140,54.083770,149.132601,166.612324,210.761530,237.655736,115.575271,128.416644,245.585955
4,2016-09-07,168.501950,245.585955,127.947061,106.523950,140.938068,144.401835,197.752337,84.829520,109.833426,83.520250
5,2016-09-10,63.253007,83.520250,51.507960,137.512950,147.829038,194.993895,209.710742,106.388291,114.361834,32.999425
6,2016-09-11,29.042917,32.999425,26.360702,100.382978,126.685031,139.257072,178.163119,78.948125,98.648366,182.608250
7,2016-09-12,132.785217,182.608250,105.117225,64.712948,102.274502,86.128249,141.872195,52.654414,80.576450,166.891925
8,2016-09-18,119.215280,166.891925,92.885558,98.749082,109.902181,134.368249,152.056209,78.885819,86.711644,130.395945
9,2016-09-19,96.302953,130.395945,76.851917,108.982181,112.230456,150.630087,155.765138,85.885688,88.255122,147.140180


Remove unusable rows

In [21]:
ewma_model_df = (
    ewma_df
    .dropna()
    .reset_index(drop=True)
)

print("Original rows:", len(ewma_df))
print("Usable rows:", len(ewma_model_df))

Original rows: 1887
Usable rows: 1885


Change/Trend features

Load data and sorting

In [22]:
change_df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

change_df["date"] = pd.to_datetime(
    change_df["date"]
)

change_df = (
    change_df
    .sort_values("date")
    .reset_index(drop=True)
)

Create target

In [23]:
change_df["target_pm10"] = (
    change_df["pm10"].shift(-1)
)

Create change features

In [24]:
for pollutant in ["pm2_5", "pm10","pm1"]:

    change_df[f"{pollutant}_change_1"] = (
        change_df[pollutant]
        - change_df[pollutant].shift(1)
    )

    change_df[f"{pollutant}_change_3"] = (
        change_df[pollutant]
        - change_df[pollutant].shift(3)
    )

    change_df[f"{pollutant}_change_7"] = (
        change_df[pollutant]
        - change_df[pollutant].shift(7)
    )

Calculate change percentage too

In [25]:
for pollutant in ["pm2_5", "pm10","pm1"]:

    change_df[f"{pollutant}_pct_change_1"] = (
        change_df[pollutant]
        .pct_change(1)
    )

    change_df[f"{pollutant}_pct_change_3"] = (
        change_df[pollutant]
        .pct_change(3)
    )

    change_df[f"{pollutant}_pct_change_7"] = (
        change_df[pollutant]
        .pct_change(7)
    )

In [26]:
display(
    change_df[
        [
            "date",
            "pm2_5",
            "pm10",
            "pm1",
            "pm2_5_change_1",
            "pm2_5_change_3",
            "pm2_5_change_7",
            "pm2_5_pct_change_1",
            "pm2_5_pct_change_3",
            "pm2_5_pct_change_7",
            "pm10_change_1",
            "pm10_change_3",
            "pm10_change_7",
            "pm10_pct_change_1",
            "pm10_pct_change_3",
            "pm10_pct_change_7",
            "pm1_change_1",
            "pm1_change_3",
            "pm1_change_7",
            "pm1_pct_change_1",
            "pm1_pct_change_3",
            "pm1_pct_change_7",
            "target_pm10",
        ]
    ].head(10)
)

,date,pm2_5,pm10,pm1,pm2_5_change_1,pm2_5_change_3,pm2_5_change_7,pm2_5_pct_change_1,pm2_5_pct_change_3,pm2_5_pct_change_7,...,pm10_pct_change_1,pm10_pct_change_3,pm10_pct_change_7,pm1_change_1,pm1_change_3,pm1_change_7,pm1_pct_change_1,pm1_pct_change_3,pm1_pct_change_7,target_pm10
0,2016-08-25,187.599837,269.346300,144.026083,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,230.975440
1,2016-08-27,159.537520,230.975440,121.881565,-28.062317,NaN,NaN,-0.149586,NaN,NaN,...,-0.142459,NaN,NaN,-22.144518,NaN,NaN,-0.153754,NaN,NaN,171.362190
2,2016-08-28,124.696523,171.362190,98.196718,-34.840997,NaN,NaN,-0.218387,NaN,NaN,...,-0.258093,NaN,NaN,-23.684847,NaN,NaN,-0.194327,NaN,NaN,78.042140
3,2016-08-29,63.915300,78.042140,54.083770,-60.781223,-123.684537,NaN,-0.487433,-0.659300,NaN,...,-0.544578,-0.710254,NaN,-44.112948,-89.942312,NaN,-0.449230,-0.624486,NaN,245.585955
4,2016-09-07,168.501950,245.585955,127.947061,104.586650,8.964430,NaN,1.636332,0.056190,NaN,...,2.146838,0.063256,NaN,73.863291,6.065496,NaN,1.365720,0.049765,NaN,83.520250
5,2016-09-10,63.253007,83.520250,51.507960,-105.248943,-61.443517,NaN,-0.624616,-0.492744,NaN,...,-0.659914,-0.512610,NaN,-76.439101,-46.688758,NaN,-0.597428,-0.475461,NaN,32.999425
6,2016-09-11,29.042917,32.999425,26.360702,-34.210090,-34.872383,NaN,-0.540845,-0.545603,NaN,...,-0.604893,-0.577159,NaN,-25.147257,-27.723068,NaN,-0.488221,-0.512595,NaN,182.608250
7,2016-09-12,132.785217,182.608250,105.117225,103.742300,-35.716733,-54.81462,3.572034,-0.211966,-0.292189,...,4.533680,-0.256439,-0.322032,78.756523,-22.829836,-38.908857,2.987649,-0.178432,-0.270151,166.891925
8,2016-09-18,119.215280,166.891925,92.885558,-13.569937,55.962273,-40.32224,-0.102195,0.884737,-0.252745,...,-0.086066,0.998221,-0.277447,-12.231668,41.377598,-28.996007,-0.116362,0.803324,-0.237903,130.395945
9,2016-09-19,96.302953,130.395945,76.851917,-22.912327,67.260037,-28.39357,-0.192193,2.315884,-0.227701,...,-0.218680,2.951461,-0.239062,-16.033640,50.491215,-21.344800,-0.172617,1.915397,-0.217368,147.140180


Remove unusable rows

In [27]:
change_model_df = (
    change_df
    .dropna()
    .reset_index(drop=True)
)

print("Original rows:", len(change_df))
print("Usable rows:", len(change_model_df))

Original rows: 1887
Usable rows: 1879


Combined one : 7-day lags, rolling mean/std, EWMA, change/trend

In [28]:
combined_df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

combined_df["date"] = pd.to_datetime(
    combined_df["date"]
)

combined_df = (
    combined_df.sort_values("date").reset_index(drop=True)
)

print("Shape:", combined_df.shape)

display(combined_df.head())

Shape: (1887, 4)


,date,pm1,pm2_5,pm10
0,2016-08-25,144.026083,187.599837,269.346300
1,2016-08-27,121.881565,159.537520,230.975440
2,2016-08-28,98.196718,124.696523,171.362190
3,2016-08-29,54.083770,63.915300,78.042140
4,2016-09-07,127.947061,168.501950,245.585955


Create target

In [29]:
combined_df["target_pm10"] = (
    combined_df["pm10"].shift(-1)
)

Add 7 days lags

In [30]:
for lag in range(1, 8):

    combined_df[f"pm2_5_lag_{lag}"] = (
        combined_df["pm2_5"].shift(lag)
    )

    combined_df[f"pm10_lag_{lag}"] = (
        combined_df["pm10"].shift(lag)
    )

Add rolling features

In [31]:
for pollutant in ["pm2_5", "pm10","pm1"]:

    combined_df[f"{pollutant}_rolling_mean_3"] = (
        combined_df[pollutant]
        .shift(1)
        .rolling(3)
        .mean()
    )

    combined_df[f"{pollutant}_rolling_mean_7"] = (
        combined_df[pollutant]
        .shift(1)
        .rolling(7)
        .mean()
    )

    combined_df[f"{pollutant}_rolling_std_7"] = (
        combined_df[pollutant]
        .shift(1)
        .rolling(7)
        .std()
    )

add ewma

In [32]:
for pollutant in ["pm2_5", "pm10","pm1"]:

    combined_df[f"{pollutant}_ewma_3"] = (
        combined_df[pollutant]
        .shift(1)
        .ewm(span=3, adjust=False)
        .mean()
    )

    combined_df[f"{pollutant}_ewma_7"] = (
        combined_df[pollutant]
        .shift(1)
        .ewm(span=7, adjust=False)
        .mean()
    )

Add change and trends

In [33]:
for pollutant in ["pm2_5", "pm10","pm1"]:

    combined_df[f"{pollutant}_ewma_3"] = (
        combined_df[pollutant]
        .shift(1)
        .ewm(span=3, adjust=False)
        .mean()
    )

    combined_df[f"{pollutant}_ewma_7"] = (
        combined_df[pollutant]
        .shift(1)
        .ewm(span=7, adjust=False)
        .mean()
    )

Feature counts

In [35]:
feature_columns = [
    col
    for col in combined_df.columns
    if col not in [
        "date",
        "pm2_5",
        "pm10",
        "pm1",
        "target_pm10",
    ]
]

print("Total features:", len(feature_columns))

print("\nFeatures:")
for feature in feature_columns:
    print(feature)

Total features: 29

Features:
pm2_5_lag_1
pm10_lag_1
pm2_5_lag_2
pm10_lag_2
pm2_5_lag_3
pm10_lag_3
pm2_5_lag_4
pm10_lag_4
pm2_5_lag_5
pm10_lag_5
pm2_5_lag_6
pm10_lag_6
pm2_5_lag_7
pm10_lag_7
pm2_5_rolling_mean_3
pm2_5_rolling_mean_7
pm2_5_rolling_std_7
pm10_rolling_mean_3
pm10_rolling_mean_7
pm10_rolling_std_7
pm1_rolling_mean_3
pm1_rolling_mean_7
pm1_rolling_std_7
pm2_5_ewma_3
pm2_5_ewma_7
pm10_ewma_3
pm10_ewma_7
pm1_ewma_3
pm1_ewma_7


Remove unusable rows

In [36]:
combined_model_df = (
    combined_df
    .dropna()
    .reset_index(drop=True)
)

print("Original rows:", len(combined_df))
print("Usable rows:", len(combined_model_df))
display(combined_model_df.head(10))
print("Features:", len(feature_columns))

Original rows: 1887
Usable rows: 1879


,date,pm1,pm2_5,pm10,target_pm10,pm2_5_lag_1,pm10_lag_1,pm2_5_lag_2,pm10_lag_2,pm2_5_lag_3,...,pm10_rolling_std_7,pm1_rolling_mean_3,pm1_rolling_mean_7,pm1_rolling_std_7,pm2_5_ewma_3,pm2_5_ewma_7,pm10_ewma_3,pm10_ewma_7,pm1_ewma_3,pm1_ewma_7
0,2016-09-12,105.117225,132.785217,182.608250,166.891925,29.042917,32.999425,63.253007,83.520250,168.501950,...,94.104586,68.605241,89.143408,45.201132,64.712948,102.274502,86.128249,141.872195,52.654414,80.576450
1,2016-09-18,92.885558,119.215280,166.891925,130.395945,132.785217,182.608250,29.042917,32.999425,63.253007,...,82.068412,60.995296,83.585000,39.339627,98.749082,109.902181,134.368249,152.056209,78.885819,86.711644
2,2016-09-19,76.851917,96.302953,130.395945,147.140180,119.215280,166.891925,132.785217,182.608250,29.042917,...,74.270861,74.787828,79.442713,36.021756,108.982181,112.230456,150.630087,155.765138,85.885688,88.255122
3,2016-09-20,80.535347,104.368733,147.140180,145.790225,96.302953,130.395945,119.215280,166.891925,132.785217,...,72.736509,91.618233,76.393456,35.060222,102.642567,108.248580,140.513016,149.422840,81.368803,85.404321
4,2016-09-21,83.568385,106.495640,145.790225,137.984515,104.368733,147.140180,96.302953,130.395945,119.215280,...,68.868714,83.424274,80.172253,33.652125,103.505650,107.278618,143.826598,148.852175,80.952075,84.187077
5,2016-09-22,82.765052,103.276103,137.984515,62.204385,106.495640,145.790225,104.368733,147.140180,96.302953,...,51.930169,80.318550,73.832442,26.591171,105.000645,107.082874,144.808412,148.086687,82.260230,84.032404
6,2016-09-23,44.094760,51.489237,62.204385,118.025195,103.276103,137.984515,106.495640,145.790225,104.368733,...,48.272617,82.289595,78.297741,24.780298,104.138374,106.131181,141.396463,145.561144,82.512641,83.715566
7,2016-09-24,72.552228,90.335197,118.025195,146.479725,51.489237,62.204385,103.276103,137.984515,106.495640,...,38.220056,70.142732,80.831178,18.760947,77.813805,92.470695,101.800424,124.721954,63.303701,73.810365
8,2016-09-25,84.983893,107.614070,146.479725,89.382590,90.335197,118.025195,51.489237,62.204385,103.276103,...,33.434839,66.470680,76.179035,15.486934,84.074501,91.936820,109.912810,123.047765,67.927964,73.495830
9,2016-09-26,53.756037,66.504697,89.382590,114.999075,107.614070,146.479725,90.335197,118.025195,51.489237,...,30.411710,67.210293,75.050226,14.309489,95.844286,95.856133,128.196267,128.905755,76.455928,76.367846


Features: 29


In [37]:
print("Missing values:")
print(
    combined_model_df[
        feature_columns + ["target_pm10"]
    ]
    .isna()
    .sum()
    .sum()
)

print("\nFinal shape:")
print(combined_model_df.shape)


Missing values:
0

Final shape:
(1879, 34)


Datasets into dictionaries

In [38]:
feature_datasets = {
    "Lag 7": df_model,
    "Rolling": rolling_model_df,
    "EWMA": ewma_model_df,
    "Change": change_model_df,
    "Combined": combined_model_df,
}

determine features

In [40]:
feature_columns_map = {}

for name, data in feature_datasets.items():

    features = [
        col
        for col in data.columns
        if col not in [
            "date",
            "pm2_5",
            "pm10",
            "pm1",
            "target_pm10",
        ]
    ]

    feature_columns_map[name] = features

    print(
        f"{name}: {len(features)} features"
    )

Lag 7: 21 features
Rolling: 24 features
EWMA: 6 features
Change: 18 features
Combined: 29 features


Chronological 80/20 split

In [42]:
splits = {}

for name, data in feature_datasets.items():

    split_index = int(len(data) * 0.8)

    train_df = data.iloc[:split_index].copy()
    val_df = data.iloc[split_index:].copy()
    features = feature_columns_map[name]
    X_train = train_df[features]
    y_train = train_df["target_pm10"]

    X_val = val_df[features]
    y_val = val_df["target_pm10"]

    splits[name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
    }

    print(
        f"{name}: "
        f"Train={len(train_df)}, "
        f"Validation={len(val_df)}"
    )

Lag 7: Train=1503, Validation=376
Rolling: Train=1503, Validation=376
EWMA: Train=1508, Validation=377
Change: Train=1503, Validation=376
Combined: Train=1503, Validation=376


Defining models

In [43]:
models = {
    "Linear Regression": LinearRegression(),

    "Ridge": Ridge(alpha=1.0),

    "Random Forest 100": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    ),

    "Random Forest 300": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),

    "Random Forest 500": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
    ),

    "Gradient Boosting 100": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),

    "Gradient Boosting 200": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),

    "Gradient Boosting 300": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),

    "Extra Trees 100": ExtraTreesRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees 300": ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees 500": ExtraTreesRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
    ),
}

Train models

In [44]:
results = []

trained_models = {}

for feature_name, split in splits.items():

    X_train = split["X_train"]
    y_train = split["y_train"]

    X_val = split["X_val"]
    y_val = split["y_val"]

    trained_models[feature_name] = {}

    print("\n" + "=" * 70)
    print(f"FEATURE SET: {feature_name}")
    print("=" * 70)

    for model_name, model in models.items():

        print(f"Training: {model_name}")

        model.fit(
            X_train,
            y_train,
        )

        predictions = model.predict(
            X_val
        )

        mae = mean_absolute_error(
            y_val,
            predictions,
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                predictions,
            )
        )

        r2 = r2_score(
            y_val,
            predictions,
        )

        results.append({
            "Feature Set": feature_name,
            "Model": model_name,
            "Features": len(
                feature_columns_map[feature_name]
            ),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
        })

        trained_models[
            feature_name
        ][model_name] = model


FEATURE SET: Lag 7
Training: Linear Regression
Training: Ridge
Training: Random Forest 100
Training: Random Forest 300
Training: Random Forest 500
Training: Gradient Boosting 100
Training: Gradient Boosting 200
Training: Gradient Boosting 300
Training: Extra Trees 100
Training: Extra Trees 300
Training: Extra Trees 500

FEATURE SET: Rolling
Training: Linear Regression
Training: Ridge
Training: Random Forest 100
Training: Random Forest 300
Training: Random Forest 500
Training: Gradient Boosting 100
Training: Gradient Boosting 200
Training: Gradient Boosting 300
Training: Extra Trees 100
Training: Extra Trees 300
Training: Extra Trees 500

FEATURE SET: EWMA
Training: Linear Regression
Training: Ridge
Training: Random Forest 100
Training: Random Forest 300
Training: Random Forest 500
Training: Gradient Boosting 100
Training: Gradient Boosting 200
Training: Gradient Boosting 300
Training: Extra Trees 100
Training: Extra Trees 300
Training: Extra Trees 500

FEATURE SET: Change
Training: Li

Create results table

In [45]:
results_df = pd.DataFrame(results)

results_df = (
    results_df
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(results_df)

,Feature Set,Model,Features,MAE,RMSE,R2
0,Change,Extra Trees 300,18,15.668077,21.224581,0.800942
1,Change,Extra Trees 100,18,15.776349,21.281336,0.799876
2,Change,Extra Trees 500,18,15.726607,21.325019,0.799053
3,Change,Random Forest 500,18,15.632944,21.710914,0.791715
4,Change,Random Forest 300,18,15.771176,21.797987,0.790041
5,Change,Random Forest 100,18,15.880982,21.934104,0.787411
6,Change,Gradient Boosting 300,18,17.839335,23.265245,0.760824
7,Lag 7,Ridge,21,16.059590,23.414185,0.757752
8,Lag 7,Linear Regression,21,16.059852,23.414873,0.757738
9,Lag 7,Gradient Boosting 100,21,16.714432,24.139203,0.742518


Rounding errors

In [46]:
results_display = results_df.copy()

results_display["MAE"] = (
    results_display["MAE"].round(3)
)

results_display["RMSE"] = (
    results_display["RMSE"].round(3)
)

results_display["R2"] = (
    results_display["R2"].round(3)
)

display(results_display)

,Feature Set,Model,Features,MAE,RMSE,R2
0,Change,Extra Trees 300,18,15.668,21.225,0.801
1,Change,Extra Trees 100,18,15.776,21.281,0.800
2,Change,Extra Trees 500,18,15.727,21.325,0.799
3,Change,Random Forest 500,18,15.633,21.711,0.792
4,Change,Random Forest 300,18,15.771,21.798,0.790
5,Change,Random Forest 100,18,15.881,21.934,0.787
6,Change,Gradient Boosting 300,18,17.839,23.265,0.761
7,Lag 7,Ridge,21,16.060,23.414,0.758
8,Lag 7,Linear Regression,21,16.060,23.415,0.758
9,Lag 7,Gradient Boosting 100,21,16.714,24.139,0.743


Best for each feature set

In [47]:
best_by_feature = (
    results_df
    .sort_values("RMSE")
    .groupby("Feature Set")
    .first()
    .reset_index()
)

display(
    best_by_feature[
        [
            "Feature Set",
            "Model",
            "Features",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
)

,Feature Set,Model,Features,MAE,RMSE,R2
0,Change,Extra Trees 300,18,15.668077,21.224581,0.800942
1,Combined,Gradient Boosting 100,29,16.571988,24.695729,0.730508
2,EWMA,Ridge,6,16.016872,24.242091,0.739951
3,Lag 7,Ridge,21,16.059590,23.414185,0.757752
4,Rolling,Gradient Boosting 100,24,17.081310,25.778796,0.706352


Overall winner

In [48]:
best_result = (
    results_df
    .sort_values("RMSE")
    .iloc[0]
)

print("BEST MODEL for data with pm1")
print("-" * 40)

print("Feature Set:", best_result["Feature Set"])
print("Model:", best_result["Model"])
print("Features:", best_result["Features"])
print("MAE:", round(best_result["MAE"], 3))
print("RMSE:", round(best_result["RMSE"], 3))
print("R²:", round(best_result["R2"], 3))

BEST MODEL for data with pm1
----------------------------------------
Feature Set: Change
Model: Extra Trees 300
Features: 18
MAE: 15.668
RMSE: 21.225
R²: 0.801
